In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
import time

# ----------------------------
# روابط الأمراض
# ----------------------------
target_diseases = {
    "Eczema": "https://dermnetnz.org/topics/atopic-dermatitis",
    "Psoriasis": "https://dermnetnz.org/topics/psoriasis-overview",
    "Vitiligo": "https://dermnetnz.org/topics/vitiligo",
    "Warts": "https://dermnetnz.org/topics/viral-wart",
    "Impetigo": "https://dermnetnz.org/topics/impetigo",
    "Acne": "https://dermnetnz.org/topics/acne-face-images",
    "Skin Cancer": "https://dermnetnz.org/topics/basal-cell-carcinoma",
    "Contact Dermatitis": "https://dermnetnz.org/topics/contact-dermatitis"
}

headers = {
    "User-Agent": "Mozilla/5.0"
}

os.makedirs("dermnet_dataset", exist_ok=True)

results = []

for name, url in target_diseases.items():
    try:
        print(f"🚀 Working on: {name}...")

        res = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, "html.parser")

        # ----------------------------
        # DESCRIPTION
        # ----------------------------
        description_parts = []
        content_block = soup.find('article') or soup.find('main') or soup

        paragraphs = content_block.find_all('p')

        for p in paragraphs:
            text = p.get_text(strip=True)

            if (
                len(text) > 60 and
                "DermNet" not in text and
                "GO TO" not in text and
                "ADVERTISEMENT" not in text
            ):
                description_parts.append(text)

            if len(description_parts) >= 10:
                break

        description = " ".join(description_parts).strip()

        # ----------------------------
        # IMAGES
        # ----------------------------
        img_tags = soup.find_all('img')

        disease_folder = f"dermnet_dataset/{name}"
        os.makedirs(disease_folder, exist_ok=True)

        saved_count = 0

        for img in img_tags:
            if saved_count >= 15:
                break

            src = img.get('src') or img.get('data-src')
            if not src:
                continue

            if any(x in src.lower() for x in ["logo", "avatar", "icon"]):
                continue

            if src.startswith('/'):
                src = "https://dermnetnz.org" + src

            try:
                img_data = requests.get(src, headers=headers, timeout=10).content

                img_filename = f"{disease_folder}/{name}_{saved_count}.jpg"

                with open(img_filename, "wb") as f:
                    f.write(img_data)

                # ----------------------------
                # IMPORTANT: ROW PER IMAGE
                # ----------------------------
                results.append({
                    "Disease": name,
                    "Image_Path": img_filename,
                    "Description": description
                })

                saved_count += 1

            except:
                continue

        print(f"✅ Saved {saved_count} images for {name}")

        time.sleep(2)

    except Exception as e:
        print(f"❌ Error in {name}: {e}")

# ----------------------------
# SAVE CSV (IMAGE-LEVEL DATASET)
# ----------------------------
df = pd.DataFrame(results)
df.to_csv("final_neural_network_data.csv", index=False)

print("🎉 Dataset created successfully!")

🚀 Working on: Eczema...
✅ Saved 15 images for Eczema
🚀 Working on: Psoriasis...
✅ Saved 2 images for Psoriasis
🚀 Working on: Vitiligo...
✅ Saved 10 images for Vitiligo
🚀 Working on: Warts...
✅ Saved 15 images for Warts
🚀 Working on: Impetigo...
✅ Saved 11 images for Impetigo
🚀 Working on: Acne...
✅ Saved 15 images for Acne
🚀 Working on: Skin Cancer...
✅ Saved 15 images for Skin Cancer
🚀 Working on: Contact Dermatitis...
✅ Saved 5 images for Contact Dermatitis
🎉 Dataset created successfully!
